In [1]:
import pandas as pd
from pathlib import Path


AUTHENTIC_ROOT = Path("dataset_split/train/authentic")
FORENSIC_ROOT = Path("dataset_split/train/forged")

material_classes = {
    "Biological",
    "Fibres",
    "Nanowires",
    "Particles",
    "Porous_Sponge",
    "Powder"
}

device_classes = {
    "Films_Coated_Surface",
    "MEMS_devices_and_electrodes",
    "Patterned_surface",
    "Tips"
}


records = []

# -----------------------------------
# Add Authentic Images
# -----------------------------------

for class_dir in AUTHENTIC_ROOT.iterdir():

    if not class_dir.is_dir():
        continue

    defect_type = class_dir.name

    if defect_type in material_classes:
        domain = "Material"
    else:
        domain = "Device"

    for img_path in class_dir.glob("*"):

        records.append({

            "image_name":
                img_path.name,

            "image_path":
                str(img_path),

            "defect_type":
                defect_type,

            "domain":
                domain,

            "authenticity":
                "Authentic",

            "mask_path":
                ""

        })


# -----------------------------------
# Add Forged Images
# -----------------------------------

for forgery_type_dir in FORENSIC_ROOT.iterdir():

    if not forgery_type_dir.is_dir():
        continue

    forgery_type = forgery_type_dir.name

    for class_dir in forgery_type_dir.iterdir():

        if not class_dir.is_dir():
            continue

        defect_type = class_dir.name

        domain = (
            "Material"
            if defect_type in material_classes
            else "Device"
        )

        for img_path in class_dir.glob("*"):

            mask_path = str(

                Path(
                    "dataset_split/train/masks"
                )

                / forgery_type
                / defect_type
                / img_path.name
            )

            records.append({

                "image_name":
                    img_path.name,

                "image_path":
                    str(img_path),

                "defect_type":
                    defect_type,

                "domain":
                    domain,

                "authenticity":
                    forgery_type,

                "mask_path":
                    mask_path
            })
# -----------------------------------
# Create DataFrame
# -----------------------------------

df = pd.DataFrame(records)

print(df.head())

# -----------------------------------
# Save CSV
# -----------------------------------

df.to_csv(
    "SEM_forensic_metadata.csv",
    index=False
)

print(
    f"\nTotal records: {len(df)}"
)

print(
    "\nSaved as SEM_forensic_metadata.csv"
)

                                image_name  \
0  L4_c0881a70509cbc98a7e8eff93d86f2f9.jpg   
1  L4_29ad5d70767fc5a51b0ce4f232fae40e.jpg   
2  L4_80ed47ecd9b707cf5210fd548f7c7286.jpg   
3  L4_a32d5b70c1374299f1866f966f3c081a.jpg   
4  L4_b3e693c0c3ab2d746d6600bcd8981c44.jpg   

                                          image_path defect_type    domain  \
0  dataset_split/train/authentic/Powder/L4_c0881a...      Powder  Material   
1  dataset_split/train/authentic/Powder/L4_29ad5d...      Powder  Material   
2  dataset_split/train/authentic/Powder/L4_80ed47...      Powder  Material   
3  dataset_split/train/authentic/Powder/L4_a32d5b...      Powder  Material   
4  dataset_split/train/authentic/Powder/L4_b3e693...      Powder  Material   

  authenticity mask_path  
0    Authentic            
1    Authentic            
2    Authentic            
3    Authentic            
4    Authentic            

Total records: 3600

Saved as SEM_forensic_metadata.csv


In [2]:
#pip install torch torchvision timm opencv-python scikit-image pillow

In [3]:
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from PIL import Image
from skimage.feature import local_binary_pattern

import timm

from torchvision.models import efficientnet_b0

import pandas as pd

import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
import os

In [4]:
from io import BytesIO
class ForensicPreprocessor:

    def __init__(self,
                 lbp_radius=2,
                 lbp_points=16):

        self.radius = lbp_radius
        self.points = lbp_points

    def normalize(self, img):

        img = img.astype(np.float32)
    
        min_val = img.min()
        max_val = img.max()
    
        if max_val - min_val < 1e-8:
            return np.zeros_like(img, dtype=np.float32)
    
        return (
            img - min_val
        ) / (
            max_val - min_val
        )

    def residual_map(self, img):

        blur = cv2.GaussianBlur(
            img,
            (5,5),
            0
        )

        residual = cv2.absdiff(
            img,
            blur
        )

        return self.normalize(residual)

    def fft_map(self, img):

        f = np.fft.fft2(img)

        fshift = np.fft.fftshift(f)

        magnitude = np.log(
            np.abs(fshift)+1
        )

        return self.normalize(magnitude)

    def lbp_map(self, img):

        lbp = local_binary_pattern(
            img,
            P=self.points,
            R=self.radius,
            method='uniform'
        )

        return self.normalize(lbp)

    def ela_map(self, img):
    
        buffer = BytesIO()
    
        Image.fromarray(img).save(
            buffer,
            format="JPEG",
            quality=90
        )
    
        buffer.seek(0)
    
        recompressed = np.array(
            Image.open(buffer).convert("L")
        )
    
        ela = cv2.absdiff(
            img,
            recompressed
        )
    
        return self.normalize(ela)

    def process(self, img):

        residual = self.residual_map(img)

        fft = self.fft_map(img)

        lbp = self.lbp_map(img)

        ela = self.ela_map(img)

        stacked = np.stack([
            residual,
            fft,
            lbp,
            ela
        ], axis=0)

        return stacked

In [5]:
class CNNBranch(nn.Module):

    def __init__(self):

        super().__init__()

        backbone = efficientnet_b0(
            weights="DEFAULT"
        )

        backbone.classifier = nn.Identity()

        self.backbone = backbone

    def forward(self,x):

        return self.backbone(x)

In [6]:
import torch.nn.functional as F

class ViTBranch(nn.Module):

    def __init__(self):
        super().__init__()

        self.vit = timm.create_model(
            'vit_base_patch16_224',
            pretrained=True,
            num_classes=0
        )

    def forward(self, x):

        x = F.interpolate(
            x,
            size=(224, 224),
            mode="bilinear",
            align_corners=False
        )

        return self.vit(x)

In [7]:
class MorphologyBranch(nn.Module):

    def __init__(self):

        super().__init__()

        self.cnn = CNNBranch()

        self.vit = ViTBranch()

        self.fc = nn.Sequential(

            nn.Linear(
                1280+768,
                1024
            ),

            nn.ReLU(),

            nn.Dropout(0.3)
        )

    def forward(self,x):

        cnn_feat = self.cnn(x)

        vit_feat = self.vit(x)

        combined = torch.cat(
            [
                cnn_feat,
                vit_feat
            ],
            dim=1
        )

        morphology_feat = self.fc(
            combined
        )

        return morphology_feat

In [8]:
class ForensicCNN(nn.Module):

    def __init__(self):

        super().__init__()

        # --------------------------------
        # Convolutional feature extractor
        # --------------------------------

        self.conv1 = nn.Sequential(

            nn.Conv2d(
                4,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.conv2 = nn.Sequential(

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.conv3 = nn.Sequential(

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        # --------------------------------
        # Global pooling for classification
        # --------------------------------

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(
            128,
            256
        )

    def forward(
        self,
        x,
        return_feature_map=False
    ):

        x = self.conv1(x)

        x = self.conv2(x)

        x = self.conv3(x)

        # --------------------------------
        # IMPORTANT:
        # Preserve spatial feature map
        # --------------------------------

        feature_map = x

        # --------------------------------
        # Classification feature
        # --------------------------------

        pooled = self.global_pool(
            feature_map
        )

        pooled = pooled.view(
            pooled.size(0),
            -1
        )

        forensic_feature = self.fc(
            pooled
        )

        if return_feature_map:

            return (
                forensic_feature,
                feature_map
            )

        return forensic_feature

In [9]:
class LocalizationDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.decoder = nn.Sequential(

            # 128 -> 64
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.Upsample(
                scale_factor=2,
                mode="bilinear",
                align_corners=False
            ),

            # 64 -> 32
            nn.Conv2d(
                64,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.Upsample(
                scale_factor=2,
                mode="bilinear",
                align_corners=False
            ),

            # 32 -> 16
            nn.Conv2d(
                32,
                16,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(16),

            nn.ReLU(),

            nn.Upsample(
                scale_factor=2,
                mode="bilinear",
                align_corners=False
            ),

            # Final mask
            nn.Conv2d(
                16,
                1,
                kernel_size=1
            )
        )

    def forward(
        self,
        feature_map,
        target_size
    ):

        x = self.decoder(
            feature_map
        )

        # Ensure exact same dimensions
        # as ground truth mask

        x = F.interpolate(
            x,
            size=target_size,
            mode="bilinear",
            align_corners=False
        )

        return x

In [10]:
class FeatureFusion(nn.Module):

    def __init__(self):
        super().__init__()

        self.morph_proj = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ReLU()
        )

        self.forensic_proj = nn.Sequential(
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.ReLU()
        )

        self.gate = nn.Sequential(
            nn.Linear(1024, 512),
            nn.Sigmoid()
        )

        self.output_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

    def forward(
        self,
        morphology_feat,
        forensic_feat
    ):

        morphology_feat = self.morph_proj(
            morphology_feat
        )

        forensic_feat = self.forensic_proj(
            forensic_feat
        )

        combined = torch.cat(
            [
                morphology_feat,
                forensic_feat
            ],
            dim=1
        )

        gate = self.gate(combined)

        fused = (
            gate * morphology_feat
            +
            (1.0 - gate) * forensic_feat
        )

        fused = self.output_fc(fused)

        return fused

In [11]:
class SEMFeatureExtractor(nn.Module):

    def __init__(self):

        super().__init__()

        self.morphology = (
            MorphologyBranch()
        )

        self.forensic = (
            ForensicCNN()
        )

        self.fusion = (
            FeatureFusion()
        )

    def forward(
        self,
        sem_image,
        forensic_maps
    ):

        # ==================================
        # Morphology branch
        # ==================================

        morphology_feat = (
            self.morphology(
                sem_image
            )
        )

        # ==================================
        # Forensic branch
        # ==================================

        (
            forensic_feat,
            forensic_feature_map
        ) = self.forensic(
            forensic_maps,
            return_feature_map=True
        )

        # ==================================
        # Gated fusion
        # ==================================

        fused_feat = self.fusion(
            morphology_feat,
            forensic_feat
        )

        return {

            "morphology_features":
                morphology_feat,

            "forensic_features":
                forensic_feat,

            "forensic_feature_map":
                forensic_feature_map,

            "fused_features":
                fused_feat
        }

In [12]:
DEFECT_MAP = {
    "Biological":0,
    "Fibres":1,
    "Films_Coated_Surface":2,
    "MEMS_devices_and_electrodes":3,
    "Nanowires":4,
    "Particles":5,
    "Patterned_surface":6,
    "Porous_Sponge":7,
    "Powder":8,
    "Tips":9
}

DOMAIN_MAP = {
    "Material":0,
    "Device":1
}

AUTH_MAP = {
    "Authentic":0,
    "copy_move":1,
    "splicing":2,
    "synthetic":3,
    "contrast":4
}

In [13]:
class SEMForensicDataset(Dataset):

    def __init__(
        self,
        csv_file,
        forensic_preprocessor
    ):

        self.df = pd.read_csv(
            csv_file
        )

        self.preprocessor = (
            forensic_preprocessor
        )

        self.transform = transforms.Compose([

            transforms.ToPILImage(),

            transforms.Grayscale(
                num_output_channels=3
            ),

            transforms.ToTensor(),

            transforms.Normalize(
                mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5]
            )
        ])

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ==================================
        # Load SEM image
        # ==================================

        img_path = row["image_path"]

        img = cv2.imread(
            img_path,
            cv2.IMREAD_GRAYSCALE
        )

        if img is None:

            raise RuntimeError(
                f"Could not read image: {img_path}"
            )

        # ==================================
        # SEM image tensor
        # ==================================

        sem_tensor = self.transform(
            img
        )

        # ==================================
        # Forensic maps
        # ==================================

        forensic_maps = (
            self.preprocessor.process(
                img
            )
        )

        forensic_tensor = torch.tensor(
            forensic_maps,
            dtype=torch.float32
        )

        # ==================================
        # Labels
        # ==================================

        defect_label = DEFECT_MAP[
            row["defect_type"]
        ]

        domain_label = DOMAIN_MAP[
            row["domain"]
        ]

        auth_label = AUTH_MAP[
            row["authenticity"]
        ]

        # ==================================
        # Ground truth manipulation mask
        # ==================================

        mask_path = str(
            row["mask_path"]
        )

        if (
            mask_path
            and mask_path != "nan"
            and os.path.exists(mask_path)
        ):

            mask = cv2.imread(
                mask_path,
                cv2.IMREAD_GRAYSCALE
            )

            if mask is None:

                raise RuntimeError(
                    f"Could not read mask: {mask_path}"
                )

            # Convert to binary
            mask = (
                mask > 127
            ).astype(np.float32)

        else:

            # Authentic images have no manipulation
            # therefore the GT mask is all zeros

            mask = np.zeros_like(
                img,
                dtype=np.float32
            )

        # ----------------------------------
        # Add channel dimension
        # ----------------------------------

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        ).unsqueeze(0)

        # ==================================
        # Return sample
        # ==================================

        sample = {

            "sem_image":
                sem_tensor,

            "forensic_maps":
                forensic_tensor,

            "mask":
                mask,

            "defect":
                torch.tensor(
                    defect_label,
                    dtype=torch.long
                ),

            "domain":
                torch.tensor(
                    domain_label,
                    dtype=torch.long
                ),

            "auth":
                torch.tensor(
                    auth_label,
                    dtype=torch.long
                )
        }

        return sample

In [14]:
preprocessor = ForensicPreprocessor()

dataset = SEMForensicDataset(

    csv_file=
    "SEM_forensic_metadata.csv",

    forensic_preprocessor=
    preprocessor
)

In [15]:
train_loader = DataLoader(

    dataset,

    batch_size=16,

    shuffle=True,

    num_workers=0
)

In [16]:
class MultiTaskClassifier(nn.Module):

    def __init__(self,
                 input_dim=512):

        super().__init__()

        # Shared representation

        self.shared = nn.Sequential(

            nn.Linear(input_dim, 256),

            nn.BatchNorm1d(256),

            nn.ReLU(),

            nn.Dropout(0.3)
        )

        # -----------------------
        # Defect Head
        # -----------------------

        self.defect_head = nn.Sequential(

            nn.Linear(256,128),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(128,10)
        )

        # -----------------------
        # Domain Head
        # -----------------------

        self.domain_head = nn.Sequential(

            nn.Linear(256,64),

            nn.ReLU(),

            nn.Linear(64,2)
        )

        # -----------------------
        # Authenticity Head
        # -----------------------

        self.auth_head = nn.Sequential(

            nn.Linear(256,128),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(128,5)
        )

    def forward(self,x):

        shared_feat = self.shared(x)

        defect_logits = self.defect_head(
            shared_feat
        )

        domain_logits = self.domain_head(
            shared_feat
        )

        auth_logits = self.auth_head(
            shared_feat
        )

        return {

            "defect_logits":
                defect_logits,

            "domain_logits":
                domain_logits,

            "auth_logits":
                auth_logits
        }

In [17]:
class ForensicSEMModel(nn.Module):

    def __init__(
        self,
        feature_extractor
    ):

        super().__init__()

        self.feature_extractor = (
            feature_extractor
        )

        self.classifier = (
            MultiTaskClassifier()
        )

        # ----------------------------------
        # Supervised localization branch
        # ----------------------------------

        # self.localization_decoder = (
        #     LocalizationDecoder()
        # )

    def forward(
        self,
        sem_image,
        forensic_maps
    ):

        features = (
            self.feature_extractor(
                sem_image,
                forensic_maps
            )
        )

        # ==================================
        # Classification
        # ==================================

        fused_features = features[
            "fused_features"
        ]

        outputs = self.classifier(
            fused_features
        )

        # ==================================
        # Localization
        # ==================================

        # feature_map = features[
        #     "forensic_feature_map"
        # ]

        # target_size = (
        #     forensic_maps.shape[-2:]
        # )

        # mask_logits = (
        #     self.localization_decoder(
        #         feature_map,
        #         target_size
        #     )
        # )

        # # ==================================
        # # Add localization output
        # # ==================================

        # outputs[
        #     "mask_logits"
        # ] = mask_logits

        return outputs

In [18]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

feature_extractor = SEMFeatureExtractor()



In [19]:
from sklearn.utils.class_weight import compute_class_weight

train_defect_labels = []
train_domain_labels = []
train_auth_labels = []

for batch in train_loader:

    train_defect_labels.extend(
        batch["defect"].cpu().numpy()
    )

    train_domain_labels.extend(
        batch["domain"].cpu().numpy()
    )

    train_auth_labels.extend(
        batch["auth"].cpu().numpy()
    )

train_defect_labels = np.array(train_defect_labels)
train_domain_labels = np.array(train_domain_labels)
train_auth_labels = np.array(train_auth_labels)

defect_classes = np.unique(train_defect_labels)

defect_weights = compute_class_weight(
    class_weight="balanced",
    classes=defect_classes,
    y=train_defect_labels
)

defect_weights = torch.tensor(
    defect_weights,
    dtype=torch.float32
).to(device)

domain_classes = np.unique(train_domain_labels)

domain_weights = compute_class_weight(
    class_weight="balanced",
    classes=domain_classes,
    y=train_domain_labels
)

domain_weights = torch.tensor(
    domain_weights,
    dtype=torch.float32
).to(device)

auth_classes = np.unique(train_auth_labels)

auth_weights = compute_class_weight(
    class_weight="balanced",
    classes=auth_classes,
    y=train_auth_labels
)

auth_weights = torch.tensor(
    auth_weights,
    dtype=torch.float32
).to(device)

In [20]:
import torch.nn.functional as F


def compute_ablation_loss(
    outputs,
    targets,
    config,
    weights,
    lambda_loc=1.0
):

    loss_dict = {}


    # ========================================================
    # AUTHENTICITY LOSS
    # ========================================================

    loss_dict["auth"] = F.cross_entropy(
        outputs["auth_logits"],
        targets["auth"]
    )


    # ========================================================
    # DEFECT LOSS
    # ========================================================

    if config["defect"]:

        loss_dict["defect"] = F.cross_entropy(
            outputs["defect_logits"],
            targets["defect"]
        )


    # ========================================================
    # DOMAIN LOSS
    # ========================================================

    if config["domain"]:

        loss_dict["domain"] = F.cross_entropy(
            outputs["domain_logits"],
            targets["domain"]
        )


    # ========================================================
    # LOCALIZATION LOSS
    # ========================================================

    # loss_dict["localization"] = localization_loss(
    #     outputs["mask_logits"],
    #     targets["mask"]
    # )


    # # ========================================================
    # # TOTAL LOSS
    # # ========================================================

    # total_loss = (
    #     lambda_loc
    #     * loss_dict["localization"]
    # )


    # Authenticity is ALWAYS active
    total_loss = (
        weights["auth"]
        * loss_dict["auth"]
    )


    # Defect only for C2/C4
    if config["defect"]:

        total_loss += (
            weights["defect"]
            * loss_dict["defect"]
        )


    # Domain only for C3/C4
    if config["domain"]:

        total_loss += (
            weights["domain"]
            * loss_dict["domain"]
        )


    return total_loss, loss_dict

In [21]:
AUTHENTIC_ROOT = Path("dataset_split/test/authentic")
FORENSIC_ROOT = Path("dataset_split/test/forged")

material_classes = {
    "Biological",
    "Fibres",
    "Nanowires",
    "Particles",
    "Porous_Sponge",
    "Powder"
}

device_classes = {
    "Films_Coated_Surface",
    "MEMS_devices_and_electrodes",
    "Patterned_surface",
    "Tips"
}


records = []

# -----------------------------------
# Add Authentic Images
# -----------------------------------

for class_dir in AUTHENTIC_ROOT.iterdir():

    if not class_dir.is_dir():
        continue

    defect_type = class_dir.name

    if defect_type in material_classes:
        domain = "Material"
    else:
        domain = "Device"

    for img_path in class_dir.glob("*"):

        records.append({

            "image_name":
                img_path.name,

            "image_path":
                str(img_path),

            "defect_type":
                defect_type,

            "domain":
                domain,

            "authenticity":
                "Authentic",

            "mask_path":
                ""

        })


# -----------------------------------
# Add Forged Images
# -----------------------------------

for forgery_type_dir in FORENSIC_ROOT.iterdir():

    if not forgery_type_dir.is_dir():
        continue

    forgery_type = forgery_type_dir.name

    for class_dir in forgery_type_dir.iterdir():

        if not class_dir.is_dir():
            continue

        defect_type = class_dir.name

        domain = (
            "Material"
            if defect_type in material_classes
            else "Device"
        )

        for img_path in class_dir.glob("*"):

            mask_path = str(

                Path(
                    "dataset_split/test/masks"
                )

                / forgery_type
                / defect_type
                / img_path.name
            )

            records.append({

                "image_name":
                    img_path.name,

                "image_path":
                    str(img_path),

                "defect_type":
                    defect_type,

                "domain":
                    domain,

                "authenticity":
                    forgery_type,

                "mask_path":
                    mask_path
            })
# -----------------------------------
# Create DataFrame
# -----------------------------------

df = pd.DataFrame(records)

print(df.head())

# -----------------------------------
# Save CSV
# -----------------------------------

df.to_csv(
    "SEM_forensic_metadata_test.csv",
    index=False
)

print(
    f"\nTotal records: {len(df)}"
)

print(
    "\nSaved as SEM_forensic_metadata.csv"
)

                                image_name  \
0  L4_31f600bbeaedd8e5864d0bdb7ae90b1a.jpg   
1  L4_87b7aae0859a54360b0ba7d96cf1a6a0.jpg   
2  L4_abcdfcc55a588be31b4d95b317820250.jpg   
3  L4_3e2bcd1e9d82cca5b97116aa46adbde2.jpg   
4  L4_1bba0cf4a9f6e655e71b4430ca0ee22d.jpg   

                                          image_path defect_type    domain  \
0  dataset_split/test/authentic/Powder/L4_31f600b...      Powder  Material   
1  dataset_split/test/authentic/Powder/L4_87b7aae...      Powder  Material   
2  dataset_split/test/authentic/Powder/L4_abcdfcc...      Powder  Material   
3  dataset_split/test/authentic/Powder/L4_3e2bcd1...      Powder  Material   
4  dataset_split/test/authentic/Powder/L4_1bba0cf...      Powder  Material   

  authenticity mask_path  
0    Authentic            
1    Authentic            
2    Authentic            
3    Authentic            
4    Authentic            

Total records: 900

Saved as SEM_forensic_metadata.csv


In [22]:
preprocessor = ForensicPreprocessor()

test_dataset = SEMForensicDataset(

    csv_file=
    "SEM_forensic_metadata_test.csv",

    forensic_preprocessor=
    preprocessor
)

In [23]:
test_loader = DataLoader(

    test_dataset,

    batch_size=16,

    shuffle=False,

    num_workers=0
)

In [24]:
import torch
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

In [25]:
import time
import torch
import numpy as np


# ============================================================
# C1-C4 ABLATION CONFIGURATIONS
# ============================================================

ABLATION_CONFIGS = {

    "C1_auth_only": {
        "defect": False,
        "domain": False,
        "auth": True
    },

    "C2_defect_auth": {
        "defect": True,
        "domain": False,
        "auth": True
    },

    "C3_domain_auth": {
        "defect": False,
        "domain": True,
        "auth": True
    },

    "C4_full_mtl": {
        "defect": True,
        "domain": True,
        "auth": True
    }
}


# ============================================================
# LOSS WEIGHTS
# ============================================================

weights = {
    "defect": 1.0,
    "domain": 1.0,
    "auth": 1.0
}

lambda_loc = 1.0

num_epochs = 50

PRINT_EVERY = 600


# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_ablation_model(
    model,
    train_loader,
    optimizer,
    scheduler,
    device,
    config_name,
    config,
    num_epochs=50,
    lambda_loc=1.0,
    weights=None,
    print_every=600
):

    if weights is None:
        weights = {
            "defect": 1.0,
            "domain": 1.0,
            "auth": 1.0
        }

    print("\n")
    print("#" * 80)
    print(f"STARTING ABLATION EXPERIMENT: {config_name}")
    print("#" * 80)

    print("\nConfiguration:")
    print(f"  Defect Loss : {config['defect']}")
    print(f"  Domain Loss : {config['domain']}")
    print(f"  Auth Loss   : {config['auth']}")
    print(f"  Localization: True")

    # --------------------------------------------------------
    # Store epoch history
    # --------------------------------------------------------

    history = {
        "loss": [],
        "defect_loss": [],
        "domain_loss": [],
        "auth_loss": []
        #"localization_loss": []
    }


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(num_epochs):

        model.train()

        epoch_start_time = time.time()

        running_loss = 0.0
        running_defect_loss = 0.0
        running_domain_loss = 0.0
        running_auth_loss = 0.0
        running_loc_loss = 0.0

        samples_processed = 0

        total_samples = len(train_loader.dataset)
        total_batches = len(train_loader)


        print("\n" + "=" * 80)

        print(
            f"[{config_name}] "
            f"Starting Epoch [{epoch+1}/{num_epochs}]"
        )

        print(
            f"Total training samples: {total_samples}"
        )

        print(
            f"Total batches: {total_batches}"
        )

        print("=" * 80)


        # ====================================================
        # BATCH LOOP
        # ====================================================

        for batch_idx, batch in enumerate(train_loader):

            batch_start_time = time.time()


            # ------------------------------------------------
            # Load batch
            # ------------------------------------------------

            sem_images = batch["sem_image"].to(device)

            forensic_maps = batch["forensic_maps"].to(device)

            defect_labels = (
                batch["defect"]
                .long()
                .to(device)
            )

            domain_labels = (
                batch["domain"]
                .long()
                .to(device)
            )

            auth_labels = (
                batch["auth"]
                .long()
                .to(device)
            )

            # mask_targets = (
            #     batch["mask"]
            #     .float()
            #     .to(device)
            # )

            batch_size = sem_images.size(0)


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            optimizer.zero_grad(set_to_none=True)

            outputs = model(
                sem_images,
                forensic_maps
            )


            # =================================================
            # TARGET DICTIONARY
            # =================================================

            targets = {
                "defect": defect_labels,
                "domain": domain_labels,
                "auth": auth_labels
                #"mask": mask_targets
            }


            # =================================================
            # ABLATION LOSS
            # =================================================

            loss, loss_dict = compute_ablation_loss(
                outputs=outputs,
                targets=targets,
                config=config,
                weights=weights,
                lambda_loc=lambda_loc
            )


            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )


            optimizer.step()


            # =================================================
            # GET INDIVIDUAL LOSSES
            # =================================================

            total_loss_value = loss.item()

            defect_loss_value = (
                loss_dict["defect"].item()
                if "defect" in loss_dict
                else 0.0
            )

            domain_loss_value = (
                loss_dict["domain"].item()
                if "domain" in loss_dict
                else 0.0
            )

            auth_loss_value = (
                loss_dict["auth"].item()
                if "auth" in loss_dict
                else 0.0
            )

            # localization_loss_value = (
            #     loss_dict["localization"].item()
            #     if "localization" in loss_dict
            #     else 0.0
            # )


            # =================================================
            # UPDATE STATISTICS
            # =================================================

            running_loss += (
                total_loss_value * batch_size
            )

            running_defect_loss += (
                defect_loss_value * batch_size
            )

            running_domain_loss += (
                domain_loss_value * batch_size
            )

            running_auth_loss += (
                auth_loss_value * batch_size
            )

            # running_loc_loss += (
            #     localization_loss_value * batch_size
            # )

            samples_processed += batch_size


            # =================================================
            # PROGRESS PRINT
            # =================================================

            if (
                samples_processed // print_every
                >
                (samples_processed - batch_size)
                // print_every

                or batch_idx == total_batches - 1
            ):

                elapsed_time = (
                    time.time() - epoch_start_time
                )

                progress = (
                    samples_processed
                    / total_samples
                ) * 100.0


                avg_loss = (
                    running_loss
                    / samples_processed
                )

                avg_defect = (
                    running_defect_loss
                    / samples_processed
                )

                avg_domain = (
                    running_domain_loss
                    / samples_processed
                )

                avg_auth = (
                    running_auth_loss
                    / samples_processed
                )

                # avg_loc = (
                #     running_loc_loss
                #     / samples_processed
                # )


                # ------------------------------------------------
                # Estimate remaining time
                # ------------------------------------------------

                if samples_processed > 0:

                    estimated_total_time = (
                        elapsed_time
                        * total_samples
                        / samples_processed
                    )

                    remaining_time = (
                        estimated_total_time
                        - elapsed_time
                    )

                else:

                    remaining_time = 0


                # ------------------------------------------------
                # Print
                # ------------------------------------------------

                print(
                    f"\n[{config_name}] "
                    f"Epoch [{epoch+1}/{num_epochs}] "
                    f"Batch [{batch_idx+1}/{total_batches}]"
                )

                print(
                    f"Samples: "
                    f"{samples_processed}/{total_samples} "
                    f"({progress:.2f}%)"
                )

                print(
                    f"Total Loss: {avg_loss:.4f} | "
                    f"Defect: {avg_defect:.4f} | "
                    f"Domain: {avg_domain:.4f} | "
                    f"Auth: {avg_auth:.4f} | "
                    #f"Localization: {avg_loc:.4f}"
                )

                print(
                    f"Elapsed: "
                    f"{elapsed_time/60:.2f} min | "
                    f"Estimated Remaining: "
                    f"{remaining_time/60:.2f} min"
                )

                print("-" * 80)


        # ====================================================
        # END OF EPOCH
        # ====================================================

        scheduler.step()


        epoch_time = (
            time.time() - epoch_start_time
        )


        epoch_loss = (
            running_loss
            / samples_processed
        )

        epoch_defect = (
            running_defect_loss
            / samples_processed
        )

        epoch_domain = (
            running_domain_loss
            / samples_processed
        )

        epoch_auth = (
            running_auth_loss
            / samples_processed
        )

        epoch_loc = (
            running_loc_loss
            / samples_processed
        )


        # ----------------------------------------------------
        # Store history
        # ----------------------------------------------------

        history["loss"].append(epoch_loss)

        history["defect_loss"].append(epoch_defect)

        history["domain_loss"].append(epoch_domain)

        history["auth_loss"].append(epoch_auth)

        # history["localization_loss"].append(epoch_loc)


        # ====================================================
        # EPOCH SUMMARY
        # ====================================================

        print("\n" + "=" * 80)

        print(
            f"[{config_name}] "
            f"Epoch [{epoch+1}/{num_epochs}] COMPLETED"
        )

        print(
            f"Average Loss       : {epoch_loss:.4f}"
        )

        print(
            f"Defect Loss        : {epoch_defect:.4f}"
        )

        print(
            f"Domain Loss        : {epoch_domain:.4f}"
        )

        print(
            f"Auth Loss          : {epoch_auth:.4f}"
        )

        print(
            f"Localization Loss  : {epoch_loc:.4f}"
        )

        print(
            f"Epoch Time         : "
            f"{epoch_time/60:.2f} minutes"
        )

        print(
            f"Learning Rate      : "
            f"{optimizer.param_groups[0]['lr']:.7f}"
        )

        print("=" * 80)


    print("\n")
    print("#" * 80)
    print(f"COMPLETED: {config_name}")
    print("#" * 80)

    return model, history

In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

import numpy as np
import torch


def evaluate_model(
        model,
        test_loader,
        device):

    model.eval()

    defect_true = []
    defect_pred = []

    domain_true = []
    domain_pred = []

    auth_true = []
    auth_pred = []


    # ============================================================
    # INFERENCE
    # ============================================================

    with torch.no_grad():

        for batch in test_loader:

            sem_images = batch[
                "sem_image"
            ].to(device)

            forensic_maps = batch[
                "forensic_maps"
            ].to(device)


            outputs = model(
                sem_images,
                forensic_maps
            )


            # ----------------------------------------------------
            # Defect
            # ----------------------------------------------------

            defect_out = torch.argmax(
                outputs["defect_logits"],
                dim=1
            )


            # ----------------------------------------------------
            # Domain
            # ----------------------------------------------------

            domain_out = torch.argmax(
                outputs["domain_logits"],
                dim=1
            )


            # ----------------------------------------------------
            # Authenticity
            # ----------------------------------------------------

            auth_out = torch.argmax(
                outputs["auth_logits"],
                dim=1
            )


            # ----------------------------------------------------
            # Store predictions
            # ----------------------------------------------------

            defect_true.extend(
                batch["defect"].cpu().numpy()
            )

            defect_pred.extend(
                defect_out.cpu().numpy()
            )


            domain_true.extend(
                batch["domain"].cpu().numpy()
            )

            domain_pred.extend(
                domain_out.cpu().numpy()
            )


            auth_true.extend(
                batch["auth"].cpu().numpy()
            )

            auth_pred.extend(
                auth_out.cpu().numpy()
            )


    # ============================================================
    # RESULTS DICTIONARY
    # ============================================================

    results = {}


    # ============================================================
    # DEFECT CLASSIFICATION
    # ============================================================

    defect_acc = accuracy_score(
        defect_true,
        defect_pred
    )


    defect_precision_w, defect_recall_w, defect_f1_w, _ = \
        precision_recall_fscore_support(
            defect_true,
            defect_pred,
            average="weighted",
            zero_division=0
        )


    defect_precision_m, defect_recall_m, defect_f1_m, _ = \
        precision_recall_fscore_support(
            defect_true,
            defect_pred,
            average="macro",
            zero_division=0
        )


    results["defect"] = {

        "accuracy": defect_acc,

        "precision_weighted":
            defect_precision_w,

        "recall_weighted":
            defect_recall_w,

        "f1_weighted":
            defect_f1_w,

        "precision_macro":
            defect_precision_m,

        "recall_macro":
            defect_recall_m,

        "f1_macro":
            defect_f1_m,

        "cm":
            confusion_matrix(
                defect_true,
                defect_pred
            )
    }


    # ============================================================
    # DOMAIN CLASSIFICATION
    # ============================================================

    domain_acc = accuracy_score(
        domain_true,
        domain_pred
    )


    domain_precision_w, domain_recall_w, domain_f1_w, _ = \
        precision_recall_fscore_support(
            domain_true,
            domain_pred,
            average="weighted",
            zero_division=0
        )


    domain_precision_m, domain_recall_m, domain_f1_m, _ = \
        precision_recall_fscore_support(
            domain_true,
            domain_pred,
            average="macro",
            zero_division=0
        )


    results["domain"] = {

        "accuracy": domain_acc,

        "precision_weighted":
            domain_precision_w,

        "recall_weighted":
            domain_recall_w,

        "f1_weighted":
            domain_f1_w,

        "precision_macro":
            domain_precision_m,

        "recall_macro":
            domain_recall_m,

        "f1_macro":
            domain_f1_m,

        "cm":
            confusion_matrix(
                domain_true,
                domain_pred
            )
    }


    # ============================================================
    # AUTHENTICITY CLASSIFICATION
    # ============================================================

    auth_acc = accuracy_score(
        auth_true,
        auth_pred
    )


    auth_precision_w, auth_recall_w, auth_f1_w, _ = \
        precision_recall_fscore_support(
            auth_true,
            auth_pred,
            average="weighted",
            zero_division=0
        )


    auth_precision_m, auth_recall_m, auth_f1_m, _ = \
        precision_recall_fscore_support(
            auth_true,
            auth_pred,
            average="macro",
            zero_division=0
        )


    results["auth"] = {

        "accuracy": auth_acc,

        "precision_weighted":
            auth_precision_w,

        "recall_weighted":
            auth_recall_w,

        "f1_weighted":
            auth_f1_w,

        "precision_macro":
            auth_precision_m,

        "recall_macro":
            auth_recall_m,

        "f1_macro":
            auth_f1_m,

        "cm":
            confusion_matrix(
                auth_true,
                auth_pred
            )
    }


    return results

In [27]:
# ============================================================
# C1-C4 ABLATION EXPERIMENT
# ============================================================

all_results = {}


for config_name, config in ABLATION_CONFIGS.items():

    print("\n\n")
    print("#" * 100)
    print(f"STARTING {config_name}")
    print("#" * 100)

     # ========================================================
    # CREATE FRESH MODEL
    # ========================================================

    model = ForensicSEMModel(
        feature_extractor
    ).to(device)


    # ========================================================
    # CREATE FRESH OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )


    # ========================================================
    # CREATE FRESH SCHEDULER
    # ========================================================

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs
    )

    # ========================================================
    # TRAIN
    # ========================================================

    trained_model, history = train_ablation_model(

        model=model,

        train_loader=train_loader,

        optimizer=optimizer,

        scheduler=scheduler,

        device=device,

        config_name=config_name,

        config=config,

        num_epochs=num_epochs,

        lambda_loc=lambda_loc,

        weights=weights,

        print_every=PRINT_EVERY
    )


    # ========================================================
    # EVALUATE IMMEDIATELY
    # ========================================================

    print("\n")
    print("=" * 100)
    print(f"EVALUATING {config_name}")
    print("=" * 100)


    results = evaluate_model(
        model=trained_model,
        test_loader=test_loader,
        device=device
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    all_results[config_name] = {

        "results": results,

        "history": history
    }


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n" + "-" * 80)

    print(f"{config_name} RESULTS")

    print("-" * 80)


    print(
        f"Defect    | "
        f"Accuracy: {results['defect']['accuracy']:.4f} | "
        f"Macro-F1: {results['defect']['f1_macro']:.4f} | "
        f"Weighted-F1: {results['defect']['f1_weighted']:.4f}"
    )


    print(
        f"Domain    | "
        f"Accuracy: {results['domain']['accuracy']:.4f} | "
        f"Macro-F1: {results['domain']['f1_macro']:.4f} | "
        f"Weighted-F1: {results['domain']['f1_weighted']:.4f}"
    )


    print(
        f"Auth      | "
        f"Accuracy: {results['auth']['accuracy']:.4f} | "
        f"Macro-F1: {results['auth']['f1_macro']:.4f} | "
        f"Weighted-F1: {results['auth']['f1_weighted']:.4f}"
    )


    print("-" * 80)


    # ========================================================
    # DELETE MODEL BEFORE NEXT ABLATION
    # ========================================================

    del trained_model
    del model
    del optimizer
    del scheduler

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\n")
print("#" * 100)
print("ALL C1-C4 ABLATION EXPERIMENTS COMPLETED")
print("#" * 100)




####################################################################################################
STARTING C1_auth_only
####################################################################################################


################################################################################
STARTING ABLATION EXPERIMENT: C1_auth_only
################################################################################

Configuration:
  Defect Loss : False
  Domain Loss : False
  Auth Loss   : True
  Localization: True

[C1_auth_only] Starting Epoch [1/50]
Total training samples: 3600
Total batches: 225

[C1_auth_only] Epoch [1/50] Batch [38/225]
Samples: 608/3600 (16.89%)
Total Loss: 1.6215 | Defect: 0.0000 | Domain: 0.0000 | Auth: 1.6215 | 
Elapsed: 1.02 min | Estimated Remaining: 5.01 min
--------------------------------------------------------------------------------

[C1_auth_only] Epoch [1/50] Batch [75/225]
Samples: 1200/3600 (33.33%)
Total Loss: 1.5590 | Defect: 0.00

In [28]:
import pandas as pd


ablation_summary = []


for config_name, experiment in all_results.items():

    results = experiment["results"]


    ablation_summary.append({

        "Configuration":
            config_name,

        # ----------------------------------------------------
        # Defect
        # ----------------------------------------------------

        "Defect Accuracy":
            results["defect"]["accuracy"],

        "Defect Macro-F1":
            results["defect"]["f1_macro"],

        "Defect Weighted-F1":
            results["defect"]["f1_weighted"],


        # ----------------------------------------------------
        # Domain
        # ----------------------------------------------------

        "Domain Accuracy":
            results["domain"]["accuracy"],

        "Domain Macro-F1":
            results["domain"]["f1_macro"],

        "Domain Weighted-F1":
            results["domain"]["f1_weighted"],


        # ----------------------------------------------------
        # Authenticity
        # ----------------------------------------------------

        "Auth Accuracy":
            results["auth"]["accuracy"],

        "Auth Macro-F1":
            results["auth"]["f1_macro"],

        "Auth Weighted-F1":
            results["auth"]["f1_weighted"]
    })


ablation_df = pd.DataFrame(
    ablation_summary
)


print("\n")
print("=" * 120)
print("C1-C4 ABLATION STUDY SUMMARY")
print("=" * 120)

print(
    ablation_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)



C1-C4 ABLATION STUDY SUMMARY
 Configuration  Defect Accuracy  Defect Macro-F1  Defect Weighted-F1  Domain Accuracy  Domain Macro-F1  Domain Weighted-F1  Auth Accuracy  Auth Macro-F1  Auth Weighted-F1
  C1_auth_only           0.0978           0.0496              0.0496           0.5811           0.4761              0.5230         0.8856         0.8866            0.8817
C2_defect_auth           0.8644           0.8646              0.8646           0.4278           0.3420              0.2944         0.8833         0.8812            0.8777
C3_domain_auth           0.1333           0.0586              0.0586           0.9322           0.9294              0.9322         0.8956         0.8939            0.8910
   C4_full_mtl           0.8711           0.8722              0.8722           0.9289           0.9257              0.9288         0.8956         0.8949            0.8914
